# Free Cursor QLoRA Training (Colab T4)\n\nThis notebook runs the full pipeline end-to-end:\n1. Verify GPU\n2. Install dependencies\n3. Mount Google Drive\n4. Extract dataset archive\n5. Train QLoRA\n6. Evaluate\n7. Merge LoRA and export ONNX bundle\n\nExpected dataset archive in Drive (fallback supported):\n- `/content/drive/MyDrive/free_cursor_dataset.tar.gz`\n- `/content/drive/MyDrive/dataset.tar.gz`

In [36]:
import os

REPO_URL = "https://github.com/dewd5252/free-cursor-mvp.git"
REPO_DIR = "/content/free-cursor-mvp"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

%cd /content/free-cursor-mvp
print("Repo ready at /content/free-cursor-mvp")

/content/free-cursor-mvp
Repo ready at /content/free-cursor-mvp


In [37]:
!nvidia-smi

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Sat Apr 18 19:59:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [38]:
!pip install -q -U transformers peft trl bitsandbytes accelerate datasets sentencepiece optimum[onnxruntime] onnx onnxruntime

In [39]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [40]:
import os
import tarfile

archive_candidates = [
    '/content/drive/MyDrive/free_cursor_dataset.tar.gz',
    '/content/drive/MyDrive/dataset.tar.gz',
]

archive_path = next((p for p in archive_candidates if os.path.exists(p)), None)
if archive_path is None:
    raise FileNotFoundError(
        'Dataset archive not found. Upload free_cursor_dataset.tar.gz to MyDrive.'
    )

print('Using archive:', archive_path)

with tarfile.open(archive_path, 'r:gz') as tar:
    tar.extractall('/content')

for required in [
    '/content/dataset/splits/train.jsonl',
    '/content/dataset/splits/val.jsonl',
    '/content/dataset/splits/test.jsonl',
]:
    if not os.path.exists(required):
        raise FileNotFoundError(f'Missing expected file: {required}')

print('Dataset extracted successfully.')

Using archive: /content/drive/MyDrive/free_cursor_dataset.tar.gz


/tmp/ipykernel_4876/1720072431.py:18: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall('/content')


Dataset extracted successfully.


In [41]:
import os
# قائمة بالملفات الموجودة في الجذر لـ Google Drive للمساعدة في تحديد مكان الملف
drive_path = '/content/drive/MyDrive'
if os.path.exists(drive_path):
    print("Files in MyDrive:")
    for f in os.listdir(drive_path):
        if f.endswith('.gz') or 'dataset' in f.lower():
            print(f"- {f}")
else:
    print("Google Drive is not mounted. Please run the mount cell above.")

Files in MyDrive:
- free_cursor_dataset.tar.gz


In [42]:
import json
import pandas as pd
from datasets import Dataset, DatasetDict

paths = {
    'train': '/content/dataset/splits/train.jsonl',
    'validation': '/content/dataset/splits/val.jsonl',
    'test': '/content/dataset/splits/test.jsonl'
}

def load_manual(path):
    data = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line))
    return Dataset.from_pandas(pd.DataFrame(data))

try:
    ds = DatasetDict({
        'train': load_manual(paths['train']),
        'validation': load_manual(paths['validation']),
        'test': load_manual(paths['test'])
    })
    print("✅ Success! Manual load completed.")
    print(ds)
    # Check first record to confirm integrity
    print("\nFirst sample target:", ds['train'][0]['target'])
except Exception as e:
    print(f"❌ Manual load failed: {e}")

✅ Success! Manual load completed.
DatasetDict({
    train: Dataset({
        features: ['sample_id', 'session_id', 'timestamp_start', 'timestamp_end', 'language', 'user_command_raw', 'user_command_normalized', 'screen_nodes', 'allowed_actions', 'context', 'target', 'quality_tags', 'source', 'redaction_flags'],
        num_rows: 48000
    })
    validation: Dataset({
        features: ['sample_id', 'session_id', 'timestamp_start', 'timestamp_end', 'language', 'user_command_raw', 'user_command_normalized', 'screen_nodes', 'allowed_actions', 'context', 'target', 'quality_tags', 'source', 'redaction_flags'],
        num_rows: 6000
    })
    test: Dataset({
        features: ['sample_id', 'session_id', 'timestamp_start', 'timestamp_end', 'language', 'user_command_raw', 'user_command_normalized', 'screen_nodes', 'allowed_actions', 'context', 'target', 'quality_tags', 'source', 'redaction_flags'],
        num_rows: 6000
    })
})

First sample target: {'action': 'type', 'app_name': None, 'co

In [48]:
import os
import json
import pandas as pd

os.makedirs('/content/dataset_cleaned/splits', exist_ok=True)

for split in ['train', 'validation', 'test']:
    df = ds[split].to_pandas()

    def robust_clean_target(t):
        if not isinstance(t, dict): return t
        # Define expected fields and their default values to force schema
        string_fields = ['text', 'direction', 'app_name', 'package_name', 'execution_mode', 'reason']
        numeric_fields = ['target_id', 'start_id', 'end_id', 'confidence']

        for k in string_fields:
            t[k] = str(t[k]) if (k in t and t[k] is not None) else ""

        for k in numeric_fields:
            if k in t and t[k] is not None:
                try:
                    t[k] = float(t[k])
                except:
                    t[k] = 0.0
            else:
                t[k] = -1.0 # Use -1.0 as a sentinel for null numeric fields
        return t

    df['target'] = df['target'].apply(robust_clean_target)

    output_path = f'/content/dataset_cleaned/splits/{split}.jsonl'
    df.to_json(output_path, orient='records', lines=True, force_ascii=False)
    print(f'✅ Processed {split}: Comprehensive cleaning applied.')

✅ Processed train: Comprehensive cleaning applied.
✅ Processed validation: Comprehensive cleaning applied.
✅ Processed test: Comprehensive cleaning applied.


In [44]:
# Final Training Command using Cleaned Data
!python scripts/ml/colab_train_qlora.py \
  --model-id Qwen/Qwen2.5-0.5B-Instruct \
  --train /content/dataset_cleaned/splits/train.jsonl \
  --val /content/dataset_cleaned/splits/validation.jsonl \
  --test /content/dataset_cleaned/splits/test.jsonl \
  --max-seq-length 1024 \
  --epochs 1 \
  --per-device-batch 4 \
  --grad-accum 4 \
  --lr 2e-4 \
  --lora-output /content/drive/MyDrive/free_cursor_lora_weights \
  --report-output /content/drive/MyDrive/free_cursor_eval_report.json

2026-04-18 20:00:42.995116: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776542443.017047   11219 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776542443.024613   11219 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776542443.043099   11219 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776542443.043152   11219 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776542443.043157   11219 computation_placer.cc:177] computation placer alr

In [45]:
!python scripts/ml/colab_train_qlora.py \
  --model-id Qwen/Qwen2.5-0.5B-Instruct \
  --train /content/dataset/splits/train.jsonl \
  --val /content/dataset/splits/val.jsonl \
  --test /content/dataset/splits/test.jsonl \
  --max-seq-length 1024 \
  --epochs 1 \
  --per-device-batch 4 \
  --grad-accum 4 \
  --lr 2e-4 \
  --lora-output /content/drive/MyDrive/free_cursor_lora_weights \
  --report-output /content/drive/MyDrive/free_cursor_eval_report.json

2026-04-18 20:01:01.916141: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776542461.936496   11328 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776542461.943249   11328 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776542461.960974   11328 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776542461.961019   11328 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776542461.961023   11328 computation_placer.cc:177] computation placer alr

In [46]:
import json

report_path = '/content/drive/MyDrive/free_cursor_eval_report.json'
with open(report_path, 'r', encoding='utf-8') as f:
    report = json.load(f)

print(json.dumps(report, ensure_ascii=False, indent=2))

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/free_cursor_eval_report.json'

In [ ]:
!python scripts/ml/merge_and_export_onnx.py \
  --base-model Qwen/Qwen2.5-0.5B-Instruct \
  --lora-dir /content/drive/MyDrive/free_cursor_lora_weights \
  --merged-dir /content/free_cursor_merged \
  --onnx-dir /content/drive/MyDrive/free_cursor_onnx_bundle

In [ ]:
import os\n\nbundle_dir = '/content/drive/MyDrive/free_cursor_onnx_bundle'\nrequired = [\n    'model.onnx',\n    'tokenizer.json',\n    'tokenizer_config.json',\n    'special_tokens_map.json',\n    'generation_config.json',\n]\n\nmissing = [name for name in required if not os.path.exists(os.path.join(bundle_dir, name))]\nif missing:\n    print('Missing files:', missing)\nelse:\n    print('ONNX bundle looks complete.')\n    for name in required:\n        path = os.path.join(bundle_dir, name)\n        print(f'- {name}: {os.path.getsize(path)} bytes')

# Final Training Execution
This cell uses the cleaned data from `/content/dataset_cleaned/` to avoid the schema mismatch error.

In [49]:
# Final Training Command
!python scripts/ml/colab_train_qlora.py \
  --model-id Qwen/Qwen2.5-0.5B-Instruct \
  --train /content/dataset_cleaned/splits/train.jsonl \
  --val /content/dataset_cleaned/splits/validation.jsonl \
  --test /content/dataset_cleaned/splits/test.jsonl \
  --max-seq-length 1024 \
  --epochs 1 \
  --per-device-batch 4 \
  --grad-accum 4 \
  --lr 2e-4 \
  --lora-output /content/drive/MyDrive/free_cursor_lora_weights \
  --report-output /content/drive/MyDrive/free_cursor_eval_report.json

2026-04-18 20:14:20.543393: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776543260.564876   14795 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776543260.572011   14795 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776543260.589957   14795 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776543260.590014   14795 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776543260.590019   14795 computation_placer.cc:177] computation placer alr

In [ ]:
import os
# Patch the training script to fix deprecated argument names
train_script = '/content/free-cursor-mvp/scripts/ml/colab_train_qlora.py'

if os.path.exists(train_script):
    with open(train_script, 'r') as f:
        content = f.read()

    # 1. Fix Transformers: evaluation_strategy -> eval_strategy
    content = content.replace('evaluation_strategy', 'eval_strategy')

    # 2. Fix TRL: tokenizer -> processing_class (for SFTTrainer)
    # We look for the keyword argument assignment specifically
    content = content.replace('tokenizer=tokenizer,', 'processing_class=tokenizer,')

    with open(train_script, 'w') as f:
        f.write(content)
    print('✅ Script patched: Fixed evaluation_strategy AND processing_class (tokenizer)')
else:
    print('❌ Script not found!')

In [51]:
# Restarting training after patch
!python scripts/ml/colab_train_qlora.py \
  --model-id Qwen/Qwen2.5-0.5B-Instruct \
  --train /content/dataset_cleaned/splits/train.jsonl \
  --val /content/dataset_cleaned/splits/validation.jsonl \
  --test /content/dataset_cleaned/splits/test.jsonl \
  --max-seq-length 1024 \
  --epochs 1 \
  --per-device-batch 4 \
  --grad-accum 4 \
  --lr 2e-4 \
  --lora-output /content/drive/MyDrive/free_cursor_lora_weights \
  --report-output /content/drive/MyDrive/free_cursor_eval_report.json

2026-04-18 20:20:26.558036: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776543626.594507   16418 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776543626.606913   16418 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776543626.640604   16418 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776543626.640642   16418 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776543626.640650   16418 computation_placer.cc:177] computation placer alr